# Experiment basic usage (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook collects the most common `Experiment` workflows in one place.
Update `system_id`, `config_dir`, and `params_dir` for your environment before you run it.

## 1. Create an `Experiment`

Create an `Experiment` first, then cache a few qubit and resonator labels for later cells.

In [1]:
import numpy as np

from IPython.display import display

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [2]:
Q0, Q1 = exp.qubit_labels[:2]
RQ0, RQ1 = exp.resonator_labels[:2]

print("qubits:", exp.qubit_labels[:4])
print("resonators:", exp.resonator_labels[:4])

qubits: ('Q00', 'Q01', 'Q02', 'Q03')
resonators: ['RQ00', 'RQ01', 'RQ02', 'RQ03']


## 2. Connect to instruments

Call `connect()` before you run measurements or schedules.

In [3]:
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Optionally push configuration to instruments

Use `configure()` only when you want to push the current configuration and parameters to the hardware.

In [4]:
# exp.configure()

## 4. Run simple waveform measurements with `measure`

Use `measure()` when you want to provide control waveforms directly and let Qubex add the readout automatically.

In [5]:
waveform = np.array(
    [
        0.01 + 0.01j,
        0.01 + 0.01j,
        0.01 + 0.01j,
        0.01 + 0.01j,
    ]
)

sequence = {
    Q0: waveform,
    Q1: waveform,
}

result = exp.measure(
    sequence=sequence,
    mode="avg",
    n_shots=1024,
)

display(result.plot())
print("avg kerneled:", result.data[Q0].kerneled)
print("avg shape:", np.asarray(result.data[Q0].kerneled).shape)

None

avg kerneled: [0.1+0.2j]
avg shape: (1,)


In [6]:
result = exp.measure(
    sequence=sequence,
    mode="single",
    n_shots=1024,
)

display(result.plot())
print("single kerneled shape:", np.asarray(result.data[Q0].kerneled).shape)
print("first 5 shots:", result.data[Q0].kerneled[:5])

None

single kerneled shape: (1024,)
first 5 shots: [0.09196674+0.20828106j 0.08785846+0.20850224j 0.14200145+0.17475473j
 0.14926317+0.18593191j 0.10387959+0.17429045j]


## 5. Build a control schedule with `PulseSchedule`

Use `PulseSchedule` when you want to build an explicit pulse sequence from reusable pulse objects.

In [7]:
pulse = qx.pulse.Gaussian(duration=64, amplitude=0.05, sigma=16)
display(pulse.plot())

schedule = qx.PulseSchedule()
with schedule as s:
    s.add(Q0, pulse)
    s.add(Q0, pulse.scaled(2))
    s.barrier()
    s.add(Q1, pulse.shifted(np.pi / 6))

display(schedule.plot())

None

None

## 6. Sweep one parameter with `sweep_parameter`

For a simple amplitude sweep, return a waveform mapping and let `Experiment` repeat the measurement over `sweep_range`.

In [8]:
result = exp.sweep_parameter(
    sequence=lambda amplitude: {
        Q0: qx.pulse.Rect(duration=64, amplitude=amplitude),
    },
    sweep_range=np.linspace(0.0, 0.1, 21),
    n_shots=1024,
    xlabel="Drive amplitude",
    ylabel="Readout response",
)

print("sweep_range:", result.data[Q0].sweep_range)
print("data shape:", np.asarray(result.data[Q0].data).shape)

sweep_range: [0.    0.005 0.01  0.015 0.02  0.025 0.03  0.035 0.04  0.045 0.05  0.055
 0.06  0.065 0.07  0.075 0.08  0.085 0.09  0.095 0.1  ]
data shape: (21,)


## 7. Sweep a `PulseSchedule` factory

A schedule factory is useful when the swept parameter changes the schedule itself, such as a wait duration.

In [9]:
wait_range = exp.util.discretize_time_range(
    np.geomspace(100, 100e3, 51),
    sampling_period=2,
)


def t1_sequence(wait: float) -> qx.PulseSchedule:
    """Build a T1-style sequence for a given wait duration."""
    schedule = qx.PulseSchedule()
    with schedule as s:
        s.add(Q0, qx.pulse.Gaussian(duration=64, amplitude=0.05, sigma=16))
        s.add(Q0, qx.pulse.Blank(duration=wait))
    return schedule


result = exp.sweep_parameter(
    sequence=t1_sequence,
    sweep_range=wait_range,
    n_shots=1024,
    xlabel="Wait duration (ns)",
    ylabel="Readout response",
    xaxis_type="log",
)

print("n_points:", len(result.data[Q0].sweep_range))
print("first 5 waits:", result.data[Q0].sweep_range[:5])

n_points: 51
first 5 waits: [100. 114. 132. 152. 174.]


## 8. Execute a schedule with explicit readout pulses

Use `execute()` when the schedule already includes the readout pulses you want to run.

In [10]:
control_pulse = qx.pulse.Gaussian(duration=64, amplitude=0.05, sigma=16)
readout_pulse = qx.pulse.FlatTop(duration=256, amplitude=0.1, tau=32)

schedule = qx.PulseSchedule()
with schedule as s:
    s.add(RQ0, readout_pulse)
    s.barrier()
    s.add(Q0, qx.pulse.Blank(duration=128))
    s.barrier()
    s.add(Q0, control_pulse)
    s.barrier()
    s.add(RQ0, readout_pulse.scaled(0.8))

display(schedule.plot())

result = exp.execute(
    schedule=schedule,
    mode="avg",
    n_shots=1024,
)

display(result.plot())
print("n_captures:", len(result.data[Q0]))

None

None

n_captures: 1


## 9. Async measurement APIs

In notebooks, call the async APIs with `await`.

In [11]:
schedule = qx.PulseSchedule()
with schedule as s:
    s.add(Q0, control_pulse)

result = await exp.run_measurement(
    schedule=schedule,
    n_shots=1024,
    shot_averaging=False,
    time_integration=True,
)

display(result.plot())
print("data shape:", result.data[Q0][0].data.shape)

None

data shape: (1,)


## 10. Async sweep measurement

Use `run_sweep_measurement()` when you want one async result for each sweep point.

In [12]:
def sweep_schedule(amplitude: float) -> qx.PulseSchedule:
    """Build a sweep schedule for one drive amplitude."""
    schedule = qx.PulseSchedule()
    with schedule as s:
        s.add(
            Q0,
            qx.pulse.Gaussian(duration=64, amplitude=amplitude, sigma=16),
        )
    return schedule


sweep_result = await exp.run_sweep_measurement(
    schedule=sweep_schedule,
    sweep_values=np.linspace(0.0, 0.1, 11),
    n_shots=1024,
    shot_averaging=False,
    time_integration=True,
)

print("n_points:", len(sweep_result.results))
print("sweep_values:", sweep_result.sweep_values)
print("data shape:", sweep_result.data[Q0][0].shape)

n_points: 11
sweep_values: [0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1 ]
data shape: (11,)


## 11. Async N-dimensional sweep measurement

Use `run_ndsweep_measurement()` when each point is identified by multiple named sweep axes.

In [13]:
def ndsweep_schedule(point) -> qx.PulseSchedule:
    """Build an ND sweep schedule for one sweep point."""
    schedule = qx.PulseSchedule()
    with schedule as s:
        s.add(
            Q0,
            qx.pulse.Gaussian(
                duration=64,
                amplitude=point["amplitude"],
                sigma=16,
            ),
        )
        s.add(Q0, qx.pulse.Blank(duration=point["wait"]))
    return schedule


nd_result = await exp.run_ndsweep_measurement(
    schedule=ndsweep_schedule,
    sweep_points={
        "amplitude": [0.02, 0.04, 0.06],
        "wait": [32, 64],
    },
    sweep_axes=("amplitude", "wait"),
    n_shots=1024,
    shot_averaging=False,
    time_integration=True,
)

print("shape:", nd_result.shape)
print("data shape:", nd_result.data[Q0][0].shape)
print("point[1, 0]:", nd_result.get_sweep_point((1, 0)))
print("n_results:", len(nd_result.results))

shape: (3, 2)
data shape: (3, 2)
point[1, 0]: {'amplitude': 0.04, 'wait': 32}
n_results: 6
